<a href="https://colab.research.google.com/github/Supimraid/DSA_Internal_Grp_4/blob/main/Models/Model_training_save.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip install pandas scikit-learn catboost xgboost seaborn matplotlib joblib sentence-transformers faiss-cpu torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 112.1 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.metrics.pairwise import cosine_similarity
from catboost import CatBoostRegressor, Pool
from sentence_transformers import SentenceTransformer
import matplotlib.pyplot as plt
import pickle
import faiss
import torch
import warnings
import os
RANDOM_STATE = 42
warnings.filterwarnings("ignore", category=UserWarning, module="huggingface_hub.utils._auth")

In [3]:
df = pd.read_csv("car_prices_extended_eda.csv", parse_dates=False)
df.columns

Index(['year', 'make', 'model', 'trim', 'body', 'transmission', 'vin', 'state',
       'condition', 'odometer', 'color', 'interior', 'seller', 'mmr',
       'sellingprice', 'saledate', 'price_diff', 'car_age', 'mileage_per_year',
       'sale_year', 'sale_month', 'sale_day', 'seller_category',
       'condition_category', 'sale_day_of_week', 'sale_day_name', 'sale_hour',
       'is_weekend', 'sale_month_name', 'price_gap_pct', 'gap_category',
       'is_luxury'],
      dtype='object')

In [4]:
drop_cols = ["vin", "saledate", "price_diff"]
df = df.drop(columns=drop_cols)

In [5]:
# Log-skewed cols (as before)
skewed_cols = ["odometer", "car_age"]
df[skewed_cols] = np.log1p(df[skewed_cols])

In [6]:
# X and y
# Transform the target to log-scale for training (log(1+x))
y = np.log1p(df["sellingprice"])
X = df.drop(columns=["sellingprice"])

In [7]:
# Auto-detect cat_features (run this once)
cat_features = X.select_dtypes(include=["object"]).columns.tolist()
print("cat_features:", cat_features)

cat_features: ['make', 'model', 'trim', 'body', 'transmission', 'state', 'color', 'interior', 'seller', 'seller_category', 'condition_category', 'sale_day_name', 'sale_month_name', 'gap_category']


In [8]:
# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [20]:
# Train with Pool (efficient)
train_pool = Pool(X_train, y_train, cat_features=cat_features)
test_pool = Pool(X_test, y_test, cat_features=cat_features) # Create test pool for eval_set
model = CatBoostRegressor(
    iterations=700,
    depth=7,
    learning_rate=0.15,
    random_seed=42,
    early_stopping_rounds=50,
    use_best_model=True,
    verbose=100
)
model.fit(train_pool, eval_set=test_pool) # Pass eval_set to fit method

0:	learn: 0.7674850	test: 0.7650243	best: 0.7650243 (0)	total: 1.13s	remaining: 13m 6s
100:	learn: 0.0385921	test: 0.0445188	best: 0.0445188 (100)	total: 1m 46s	remaining: 10m 34s
200:	learn: 0.0340853	test: 0.0426105	best: 0.0426105 (200)	total: 3m 28s	remaining: 8m 38s
300:	learn: 0.0312159	test: 0.0418277	best: 0.0418140 (297)	total: 5m 14s	remaining: 6m 56s
400:	learn: 0.0292171	test: 0.0414880	best: 0.0414880 (400)	total: 7m	remaining: 5m 13s
500:	learn: 0.0275113	test: 0.0412035	best: 0.0411957 (499)	total: 8m 44s	remaining: 3m 28s
600:	learn: 0.0258203	test: 0.0410834	best: 0.0410834 (600)	total: 10m 30s	remaining: 1m 43s
699:	learn: 0.0244397	test: 0.0409785	best: 0.0409299 (666)	total: 12m 15s	remaining: 0us

bestTest = 0.04092990983
bestIteration = 666

Shrink model to first 667 iterations.


In [21]:
# Make predictions on the log scale
y_train_pred_log = model.predict(X_train)
y_test_pred_log = model.predict(X_test)

# --- Back-Transformation ---
# y is already log-transformed (y = np.log1p(df["sellingprice"])), so we just index it.
y_train_actual_log = y.loc[X_train.index]
y_test_actual_log = y.loc[X_test.index]

# 1. Back-Transform predictions and actuals to dollar scale using np.expm1 (e^x - 1)
y_train_actual_dollars = np.expm1(y_train_actual_log)
y_test_actual_dollars = np.expm1(y_test_actual_log)
y_train_pred_dollars = np.expm1(y_train_pred_log)
y_test_pred_dollars = np.expm1(y_test_pred_log)

# --- Calculate Metrics (MAE, RMSE) on the DOLLAR SCALE ---
train_mae = mean_absolute_error(y_train_actual_dollars, y_train_pred_dollars)
test_mae = mean_absolute_error(y_test_actual_dollars, y_test_pred_dollars)
train_rmse = np.sqrt(mean_squared_error(y_train_actual_dollars, y_train_pred_dollars))
test_rmse = np.sqrt(mean_squared_error(y_test_actual_dollars, y_test_pred_dollars))

# Create a dictionary to hold the results
results = {
    'Metric': ['Mean Absolute Error (MAE)', 'Root Mean Squared Error (RMSE)'],
    'Train Result (USD)': [f'${train_mae:,.0f}', f'${train_rmse:,.0f}'],
    'Test Result (USD)': [f'${test_mae:,.0f}', f'${test_rmse:,.0f}'],
    'Train/Test Gap': [f'{((train_mae - test_mae) / test_mae * 100):+.1f}%',
                     f'{((train_rmse - test_rmse) / test_rmse * 100):+.1f}%']
}

# Create a DataFrame for presentation
df_results = pd.DataFrame(results)

# Print title and results
print("--- Model Performance Metrics (on Dollar Scale) ---")
print(df_results.to_markdown(index=False)) # Use markdown for clean printing in notebooks/markdown environments

# Check for Overfitting
if test_rmse / train_rmse > 1.15:
    print("\n⚠️ Significant Overfitting: Test RMSE is 15%+ worse than Train RMSE.")
elif test_rmse / train_rmse < 0.85:
    print("\n⚠️ Potential Data Leakage: Test RMSE is much better than Train RMSE.")
else:
    print("\n✅ Good Fit: Metrics are comparable, model generalizes well.")

--- Model Performance Metrics (on Dollar Scale) ---
| Metric                         | Train Result (USD)   | Test Result (USD)   | Train/Test Gap   |
|:-------------------------------|:---------------------|:--------------------|:-----------------|
| Mean Absolute Error (MAE)      | $117                 | $120                | -2.6%            |
| Root Mean Squared Error (RMSE) | $846                 | $880                | -3.9%            |

✅ Good Fit: Metrics are comparable, model generalizes well.


In [22]:
# Assuming 'model' is the CatBoostRegressor object trained in the previous cell.

MODEL_FILENAME = 'car_price_catboost_model.cbm'

# 1. Save the model to a binary CBM file
model.save_model(
    MODEL_FILENAME,
    format='cbm' # 'cbm' is the default binary format
)

print(f"✅ Model saved successfully as: {MODEL_FILENAME}")

✅ Model saved successfully as: car_price_catboost_model.cbm
